# 05 · Data cleaning: Market Bucks (`marketbucksUsage.xlsx`)

**What Market Bucks is** (data dictionary): a nutrition incentive at participating **farmers markets**. For every $1 of SNAP (EBT) a customer spends, they get $1 in Market Bucks, up to $10 per visit. `PB` = Produce Bucks, a separate grant-funded incentive; `RC` = Ramsey County-specific incentives.

**Input:** `Data/marketbucksUsage.xlsx` (one row per market per date entry, Jan 2022 – Jul 2026) and `Data/processed/organization_clean.csv` (market locations, from notebook 04).

**Outputs (`Data/processed/`):**
1. `marketbucks_market_day.csv`: one row per market per market day.
2. `marketbucks_county_month.csv`: totals per county per month (located markets only).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
OUT_DIR = DATA / "processed"
mmg = pd.read_csv(OUT_DIR / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})
COUNTY_FIPS = dict(zip(mmg["county"], mmg["fips"]))      # official 87 counties -> FIPS
FIPS_COUNTY = {v: k for k, v in COUNTY_FIPS.items()}

raw = pd.read_excel(DATA / "marketbucksUsage.xlsx")
org = pd.read_csv(OUT_DIR / "organization_clean.csv", dtype={"fips": str, "zip": str, "block_geoid": str})
print("raw:", raw.shape, "| entry id unique:", raw["_ID"].is_unique, "| markets:", raw["_id_site"].nunique())

raw: (13573, 19) | entry id unique: True | markets: 155


## Step 1: Columns, dates and money fields

- **What:** keep the market ID, date and all dollar / count fields, renamed. Blank money fields mean **nothing was recorded**; we keep them blank rather than 0 for single entries.
- **How to read the fields:**
  - **Distributed** = value handed out or exchanged that day.
  - **Redeemed** = value actually spent with vendors. Redeemed can **exceed** distributed, because tokens from earlier visits can be spent later (data dictionary). That is not an error.

In [2]:
COLS = {"_ID": "entry_id", "_id_site": "site_id", "Dated": "date",
        "SNAPDistributed": "snap_distributed", "SNAPRedeemed": "snap_redeemed",
        "MBDistributed": "mb_distributed", "MBRedeemed": "mb_redeemed",
        "PBDistributed": "pb_distributed", "PBRedeemed": "pb_redeemed",
        "RCMBDistributed": "rc_mb_distributed", "RCMBRedeemed": "rc_mb_redeemed",
        "GiftCerts": "gift_certs", "RCGiftCertificates": "rc_gift_certs",
        "Customers": "customers", "Transactions": "transactions", "isLateEntry": "is_late_entry"}
MONEY = ["snap_distributed", "snap_redeemed", "mb_distributed", "mb_redeemed", "pb_distributed", "pb_redeemed",
         "rc_mb_distributed", "rc_mb_redeemed", "gift_certs", "rc_gift_certs"]
COUNTS = ["customers", "transactions"]
mb = raw[list(COLS)].rename(columns=COLS)
mb["date"] = pd.to_datetime(mb["date"])
mb["is_late_entry"] = mb["is_late_entry"].eq(1)
print("dates:", mb["date"].min().date(), "->", mb["date"].max().date())
print("negative values:", int((mb[MONEY + COUNTS] < 0).sum().sum()))
print("entries where redeemed > distributed (allowed: carry-over):",
      {"SNAP": int((mb["snap_redeemed"] > mb["snap_distributed"] + 0.01).sum()),
       "Market Bucks": int((mb["mb_redeemed"] > mb["mb_distributed"] + 0.01).sum())})

dates: 2022-01-01 -> 2026-07-31
negative values: 0
entries where redeemed > distributed (allowed: carry-over): {'SNAP': 2812, 'Market Bucks': 3514}


## Step 2: Duplicate entries for the same market day

- **What:**
  1. Drop **exact copies**: same market, date and every value.
  2. Where a market has **several different entries for one day**, **add them together** into one market-day row and record how many entries were combined (`n_entries`).
- **Why:** the different-value entries look like separate tally sheets for one day (e.g. one entry of $52.61 and another of $0). Keeping only one would lose real spending. The same approach was used for food shelf duplicates.

In [3]:
exact = mb.duplicated(["site_id", "date"] + MONEY + COUNTS, keep="first")
print(f"exact duplicate entries removed: {int(exact.sum())}")
mb = mb[~exact]

multi = mb.duplicated(["site_id", "date"], keep=False)
print(f"market days with several different entries: {mb.loc[multi, ['site_id', 'date']].drop_duplicates().shape[0]} "
      f"({int(multi.sum())} entries)")
day = (mb.groupby(["site_id", "date"])
         .agg(**{c: (c, lambda s: s.sum(min_count=1)) for c in MONEY + COUNTS},
              n_entries=("entry_id", "size"), is_late_entry=("is_late_entry", "any"))
         .reset_index())
print("market-day rows:", len(day))

exact duplicate entries removed: 25
market days with several different entries: 228 (482 entries)


market-day rows: 13294


## Step 3: Data-quality flags

Flags mark rows to be aware of; nothing is deleted.
- `flag_all_zero`: every value is 0 or blank (a market day with no recorded SNAP activity).
- `flag_customers_unreliable`: `customers` is 0 or blank although `transactions` > 0. This happens in **about half** of all rows, and yearly customer totals jump (≈6,400 → 3,600 → 8,500), so **use `transactions`, not `customers`, as the participation count**.
- `flag_outlier`: SNAP redeemed is more than 10× that market's median day (and over $1,000). Kept, but worth checking.

In [4]:
filled = day[MONEY + COUNTS].fillna(0)
day["flag_all_zero"] = filled.sum(axis=1) == 0
day["flag_customers_unreliable"] = (day["customers"].fillna(0) == 0) & (day["transactions"].fillna(0) > 0)
site_median = day.groupby("site_id")["snap_redeemed"].transform("median")
day["flag_outlier"] = (day["snap_redeemed"] > 10 * site_median) & (day["snap_redeemed"] > 1000)
print({f: int(day[f].sum()) for f in ["flag_all_zero", "flag_customers_unreliable", "flag_outlier"]})
day.groupby(day["date"].dt.year)[["customers", "transactions", "snap_redeemed", "mb_redeemed"]].sum().round(0)

{'flag_all_zero': 319, 'flag_customers_unreliable': 6206, 'flag_outlier': 21}


,customers,transactions,snap_redeemed,mb_redeemed
date,,,,
2022,6404.0,36910.0,551429.0,315357.0
2023,3631.0,49301.0,563603.0,334784.0
2024,8531.0,46746.0,586098.0,370561.0
2025,4046.0,44552.0,568550.0,388460.0
2026,2350.0,17013.0,191694.0,111384.0


## Step 4: Market location (county + FIPS)

- **What:** attach each market's county, FIPS and map location from `organization_clean.csv`.
- **Known gap:** some markets have **no address, city, ZIP or block** in `Organization.xlsx`, so their county can't be determined. They are **kept** (they still count in statewide totals), marked `county = NaN`, and left out of the county-month table.

In [5]:
loc = org[["org_id", "county", "fips", "city", "zip", "lat", "lon", "status", "food_bank"]].rename(columns={"org_id": "site_id"})
day["site_id"] = day["site_id"].astype(float)      # same type as org_id (which can have decimals)
day = day.merge(loc, on="site_id", how="left", validate="many_to_one")
no_loc = day["fips"].isna()
print(f"markets: {day['site_id'].nunique()} | without a county: {day.loc[no_loc, 'site_id'].nunique()} "
      f"| their market days: {int(no_loc.sum())} ({no_loc.mean():.1%}) "
      f"| their share of SNAP redeemed: {day.loc[no_loc, 'snap_redeemed'].sum() / day['snap_redeemed'].sum():.1%}")
print("counties with at least one located market:", day["fips"].nunique())

markets: 155 | without a county: 31 | their market days: 738 (5.6%) | their share of SNAP redeemed: 0.9%
counties with at least one located market: 51


## Step 5: Save

In [6]:
day["year"] = day["date"].dt.year
day["month"] = day["date"].dt.to_period("M").dt.to_timestamp()
ORDER = (["site_id", "date", "year", "month", "county", "fips", "city", "zip", "lat", "lon", "status", "food_bank"]
         + MONEY + COUNTS + ["n_entries", "is_late_entry", "flag_all_zero", "flag_customers_unreliable", "flag_outlier"])
day = day[ORDER].sort_values(["site_id", "date"]).reset_index(drop=True)
assert not day.duplicated(["site_id", "date"]).any()
day.to_csv(OUT_DIR / "marketbucks_market_day.csv", index=False)

county_month = (day[day["fips"].notna()]
                .groupby(["fips", "county", "month"])
                .agg(n_markets=("site_id", "nunique"), n_market_days=("site_id", "size"),
                     **{c: (c, "sum") for c in ["snap_redeemed", "mb_distributed", "mb_redeemed", "pb_distributed",
                                                "pb_redeemed", "transactions"]})
                .reset_index())
county_month.to_csv(OUT_DIR / "marketbucks_county_month.csv", index=False)
print("saved marketbucks_market_day.csv:", day.shape, "| marketbucks_county_month.csv:", county_month.shape)
print("market days by month of year (seasonality):", day["date"].dt.month.value_counts().sort_index().to_dict())

saved marketbucks_market_day.csv: (13294, 29) | marketbucks_county_month.csv: (1402, 11)
market days by month of year (seasonality): {1: 278, 2: 298, 3: 338, 4: 303, 5: 987, 6: 1992, 7: 2630, 8: 2304, 9: 2095, 10: 1504, 11: 315, 12: 250}


## Summary

- **`marketbucks_market_day.csv`:** 13,294 market days at 155 markets, Jan 2022 – Jul 2026.
- **Cleaning:** 25 exact duplicate entries removed; 228 market days with several different entries were added together (`n_entries`). No negative values.
- **Seasonal:** most activity runs May–October.
- **Use `transactions`, not `customers`:** `customers` is 0 or blank in 6,206 rows that had transactions (flagged).
- **Other flags:** 21 outlier days (kept, flagged); 319 days with no recorded activity.
- **31 markets have no location** (738 market days, 5.6% of rows, only 0.9% of SNAP dollars). They are kept in the market-day file but not in `marketbucks_county_month.csv` (1,402 rows, 51 counties).
- **Scale:** about $0.55–0.59M of SNAP spent per year at these markets, small next to food shelf activity.